# Subquestion 4
Is the extension of the free student travel product to MBO students under 18 (1 January 2017) associated with an increase in train and bus/tram/metro use among 12–18 year olds, compared with 18–25 year olds?

In [43]:
# Loading packages
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import nbformat
idx = pd.IndexSlice

In [44]:
# Loading the data sets
path2 = '../Data_age_research/2_Mobility_per_person_by_transport_mode_and_purpose_(84710NED).csv'
path4 = '../Data_age_research/4_Mobility_per_person_by_personal_characteristics_(84709NED).csv'

data_set2 = pd.read_csv(path2, sep=";", decimal=',', skiprows=5, header=[0, 1, 2, 3], index_col=0, skipfooter=1, na_values=['.', ''], engine='python')
data_set4 = pd.read_csv(path4, sep=';', decimal=',', skiprows=7, header=[0, 1, 2], index_col=0, na_values=['.', ''], engine='python')

In [45]:
# display(data_set2)

agegroups = ['12 to 18 years', '18 to 25 years']
transportmode = ['Train', 'Bus/tram/metro']

data2_filtered = data_set2.loc[:, idx[agegroups, transportmode, 'Total', :]]
display(data2_filtered)

Age                12 to 18 years                18 to 25 years               
Modes of transport          Train Bus/tram/metro          Train Bus/tram/metro
Travel purposes             Total          Total          Total          Total
Periods                    number         number         number         number
1999*                        0.07           0.23           0.21           0.43
2000*                        0.07           0.22           0.22           0.43
2001*                        0.07           0.22           0.22           0.42
2002*                        0.07           0.21           0.21           0.41
2003*                        0.07           0.20           0.21           0.41
2004*                        0.06           0.20           0.21           0.40
2005*                        0.06           0.19           0.22           0.40
2006*                        0.06           0.19           0.22           0.40
2007*                        0.06           0.19           0.23           0.40
2008*                        0.06           0.18           0.23           0.39
2009*                        0.06           0.18           0.23           0.40
2010*                        0.06           0.18           0.23           0.40
2011*                        0.06           0.18           0.24           0.40
2012*                        0.06           0.17           0.24           0.39
2013*                        0.06           0.18           0.25           0.40
2014*                        0.06           0.18           0.25           0.39
2015*                        0.06           0.18           0.25           0.39
2016*                        0.07           0.18           0.24           0.39
2017*                        0.07           0.18           0.25           0.39
2018*                        0.07           0.19           0.24           0.38
2019*                        0.07           0.18           0.25           0.38
2020*                        0.03           0.10           0.13           0.20
2021*                        0.04           0.11           0.14           0.21
2022*                        0.05           0.14           0.18           0.27
2023*                        0.06           0.15           0.20           0.31
2024*                        0.06           0.16           0.21           0.31
2025*                        0.06           0.15           0.21           0.31

In [ ]:
# Clean index years
data_set2.index = data_set2.index.str.replace('*', '', regex=False).astype(int)

# Filter the data
df2_train = data_set2.loc[:, idx[agegroups, 'Train', 'Total', 'number']]
df2_btm = data_set2.loc[:, idx[agegroups, 'Bus/tram/metro', 'Total', 'number']]

# Simplify column names to age group level
df2_train.columns = df2_train.columns.get_level_values('Age')
df2_btm.columns = df2_btm.columns.get_level_values('Age')

# Calculate total public transport (PT) use
df2_total_pt = df2_train + df2_btm

# Filter data from year 2000 onwards
df2_total_pt = df2_total_pt.loc[2000:]

# Index values relative to 2016 (2016 = 100%)
df2_indexed = (df2_total_pt / df2_total_pt.loc[2016]) * 100

# Create figure with 2 stacked subplots sharing the x-axis
fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True, 
    vertical_spacing=0.1,
    subplot_titles=(
        'A) Absolute Total PT Use (Train + Bus/Tram/Metro)', 
        'B) Relative Change vs. Reference Year 2016 (Index = 100%)'
    )
)

# Line colors
color_12_18 = '#1f77b4'
color_18_25 = '#ff7f0e'

# --- Subplot A: Absolute numbers ---
fig.add_trace(
    go.Scatter(
        x=df2_total_pt.index, 
        y=df2_total_pt['12 to 18 years'],
        mode='lines+markers',
        name='12–18 years',
        line=dict(color=color_12_18, width=2),
        marker=dict(symbol='circle', size=7),
        legendgroup='12_18'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=df2_total_pt.index, 
        y=df2_total_pt['18 to 25 years'],
        mode='lines+markers',
        name='18–25 years',
        line=dict(color=color_18_25, width=2),
        marker=dict(symbol='square', size=7),
        legendgroup='18_25'
    ),
    row=1, col=1
)

# Policy change vertical line for Subplot A
fig.add_vline(
    x=2017, line_dash="dash", line_color="red", line_width=1.5,
    row=1, col=1
)

# --- Subplot B: Indexed trend ---
fig.add_trace(
    go.Scatter(
        x=df2_indexed.index, 
        y=df2_indexed['12 to 18 years'],
        mode='lines+markers',
        name='12–18 years',
        line=dict(color=color_12_18, width=2),
        marker=dict(symbol='circle', size=7),
        legendgroup='12_18',
        showlegend=False
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=df2_indexed.index, 
        y=df2_indexed['18 to 25 years'],
        mode='lines+markers',
        name='18–25 years',
        line=dict(color=color_18_25, width=2),
        marker=dict(symbol='square', size=7),
        legendgroup='18_25',
        showlegend=False
    ),
    row=2, col=1
)

# Policy change vertical line with annotation for Subplot B
fig.add_vline(
    x=2017, line_dash="dash", line_color="red", line_width=1.5,
    annotation_text="Policy change (Jan 1, 2017)",
    annotation_position="top left",
    row=2, col=1
)

# Axis labels and layout formatting
fig.update_yaxes(title_text="Trips p.p.p.d.", row=1, col=1)
fig.update_yaxes(title_text="Index (2016 = 100%)", row=2, col=1)
fig.update_xaxes(title_text="Year", row=2, col=1)

fig.update_layout(
    height=750,
    width=900,
    template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)

fig.show()